# KidsNutriBite - Experiment 1: Qwen2.5-7B-Instruct WITHOUT RAG (Groq judge)

**Standalone research-evaluation notebook - simplified, single-purpose.** This notebook obtains only the missing piece of Experiment 1: Faithfulness, Answer Relevancy, and Hallucination Rate for **Qwen2.5-7B-Instruct WITHOUT RAG**, evaluated with the project's real, unmodified production Groq judge infrastructure - the exact same judge configuration (`model_name="groq_judge"` -> `openai/gpt-oss-120b`) and the exact same metric formulas (`evaluation/metrics/grounding_metrics.py`, `evaluation/metrics/relevancy_metrics.py`, unmodified) used by the official Qwen+RAG benchmark.

**What this notebook does NOT do:**
- Does **not** call NVIDIA/DeepSeek anywhere - no `NvidiaJudgeClient`, no `openai` package, no NVIDIA API.
- Does **not** re-run the Qwen+RAG evaluation and does **not** recompute the RAG-side metrics - those already exist from the official benchmark and are loaded from a single hardcoded, clearly-labeled cell (Section 10), never estimated.
- Does **not** regenerate any Qwen answers - `config_a_results` (the 30 already-generated no-RAG answers) is produced once by Section 8 exactly as before and is never regenerated afterward.
- Does **not** change `evaluation/metrics/grounding_metrics.py`, `evaluation/metrics/relevancy_metrics.py`, or any production judge class.

**Judge:** the project's real `KidsNutriLLMClient` (`llm/llm_client.py`, unmodified) with `model_name="groq_judge"` - the identical class instance already used for Qwen generation in this notebook is reused directly as the judge's `llm_client`, exactly as `evaluation/evaluator.py` does for the official benchmark. No notebook-local adapter class is created.

**Fair-metric design (unchanged from the original Experiment 1 design):** the no-RAG configuration's Faithfulness is a **baseline factual-grounding evaluation** - claims are checked against this benchmark's verified `gold_facts`/`reference_answer` (`evidence_source = "gold_reference_baseline"`), never against an empty RAG context. This is a genuinely different evidence source from the RAG configuration's retrieved-context faithfulness, so the two numbers are documented as such, not treated as identical measures.

**Checkpointing:** the 30-case Groq evaluation loop (Section 14) saves its running results to `reports/experiment1_no_rag_checkpoint.json` after **every single case** - an interrupted run (rate limit, kernel restart, network drop) can be re-run from the top and will skip every case already present in the checkpoint instead of re-spending Groq quota on them.

**Failure handling:** identical philosophy to the production evaluator - a judge failure is never converted into a score of 0. It is recorded explicitly (`*_status` columns, `*_judge_failed` flags) and excluded from averages.

**Outputs:** `experiment1_no_rag_metrics.csv`, `experiment1_no_rag_answers.csv`, `experiment1_no_rag_summary.json`, `experiment1_rag_vs_no_rag_metrics.csv`, `experiment1_metric_comparison.png`, `experiment1_hallucination_rate.png` - all written to `reports/` under the extracted project root.

## Section 1: Environment / GPU

Confirm this kernel is running on Kaggle, detect CUDA, and confirm a T4 (or at least *some* CUDA GPU) is attached before anything else runs.

In [ ]:
import os
import torch

print("=== Environment / GPU ===")
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
print(f"Running on Kaggle: {'YES' if ON_KAGGLE else 'NO'}")

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU name: {gpu_name}")
    print(f"Total VRAM: {total_vram:.2f} GB")
    if "T4" in gpu_name:
        print("Confirmed: NVIDIA T4 detected.")
    else:
        print(f"WARNING: expected an NVIDIA T4, detected '{gpu_name}' instead. "
              f"4-bit Qwen inference should still work on any CUDA GPU with enough VRAM, "
              f"but timings/behavior may differ from the approved T4 baseline.")
else:
    print("WARNING: no CUDA GPU detected. Qwen local generation will raise a RuntimeError "
          "later in this notebook (Section 7). On Kaggle: Settings -> Accelerator -> GPU T4 x2, "
          "then re-run this notebook from the top.")

## Section 2: Kaggle Project Dataset Discovery + Extraction

The approved project snapshot is attached to this notebook as a Kaggle Dataset, mounted read-only under `/kaggle/input/<dataset-name>/`. This cell locates it, then extracts/copies it into a writable directory under `/kaggle/working/`, since this notebook needs to write reports and cached model files.

Set `KAGGLE_PROJECT_DATASET` below to the exact slug of the dataset you attached (visible in the notebook's "Data" pane). If it isn't found, the cell falls back to the single dataset under `/kaggle/input/` when there is exactly one attached - it never guesses among several.

In [ ]:
# EDIT this to match the Kaggle Dataset you attached to this notebook.
KAGGLE_PROJECT_DATASET = "nutrikid-agentic"

import glob
import shutil
import zipfile

KAGGLE_INPUT_ROOT = "/kaggle/input"
KAGGLE_WORKING_ROOT = "/kaggle/working"
PROJECT_DIR_NAME = "kidsnutribite_experiment1_project"


def _find_input_dataset_dir():
    candidate = os.path.join(KAGGLE_INPUT_ROOT, KAGGLE_PROJECT_DATASET)
    if os.path.isdir(candidate):
        return candidate
    if os.path.isdir(KAGGLE_INPUT_ROOT):
        subdirs = [d for d in glob.glob(os.path.join(KAGGLE_INPUT_ROOT, "*")) if os.path.isdir(d)]
        if len(subdirs) == 1:
            print(f"NOTE: '{KAGGLE_PROJECT_DATASET}' not found under {KAGGLE_INPUT_ROOT}; "
                  f"exactly one dataset is attached, using it instead: {subdirs[0]}")
            return subdirs[0]
    return None


if ON_KAGGLE:
    input_dataset_dir = _find_input_dataset_dir()
    if input_dataset_dir is None:
        raise FileNotFoundError(
            f"No Kaggle dataset found under {KAGGLE_INPUT_ROOT}. Attach the approved project "
            f"snapshot as a Kaggle Dataset and/or set KAGGLE_PROJECT_DATASET above to match its slug."
        )
    print(f"Detected input dataset path: {input_dataset_dir}")

    project_root = os.path.join(KAGGLE_WORKING_ROOT, PROJECT_DIR_NAME)
    if os.path.exists(project_root):
        shutil.rmtree(project_root)

    if os.path.exists(os.path.join(input_dataset_dir, "main.py")):
        # Kaggle already extracted the dataset zip into a plain project tree.
        shutil.copytree(input_dataset_dir, project_root)
    else:
        zip_candidates = glob.glob(os.path.join(input_dataset_dir, "**", "*.zip"), recursive=True)
        if not zip_candidates:
            raise FileNotFoundError(
                f"'{input_dataset_dir}' contains neither a main.py project tree nor a .zip snapshot."
            )
        zip_path = zip_candidates[0]
        print(f"Extracting project snapshot: {zip_path}")
        extract_dir = os.path.join(KAGGLE_WORKING_ROOT, "_kidsnutribite_exp1_extracted")
        if os.path.exists(extract_dir):
            shutil.rmtree(extract_dir)
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)
        entries = [e for e in os.listdir(extract_dir) if not e.startswith("__MACOSX")]
        source_root = (
            os.path.join(extract_dir, entries[0])
            if len(entries) == 1 and os.path.isdir(os.path.join(extract_dir, entries[0]))
            else extract_dir
        )
        shutil.copytree(source_root, project_root)

    print(f"Extracted project path: {project_root}")
else:
    project_root = os.getcwd()
    print("Not running on Kaggle (KAGGLE_KERNEL_RUN_TYPE not set) - using the current directory as the project root.")
    print(f"Project path: {project_root}")

print(f"Current working directory (before chdir): {os.getcwd()}")
assert os.path.exists(os.path.join(project_root, "main.py")), (
    f"main.py not found under '{project_root}' - dataset extraction/copy did not produce a valid project tree."
)
print("Confirmed: project root contains main.py")

## Section 3: Dependency Installation

Installs exactly what THIS notebook needs. `groq` is required for the Groq judge (the project's official primary judge provider). `sentence-transformers` is required only to construct the same embedding model (`BAAI/bge-small-en-v1.5`) the official benchmark uses for Answer Relevancy - not for retrieval, since this notebook never runs RAG retrieval. `faiss-cpu`/`rank-bm25` are deliberately **not** installed - no FAISS index or BM25 retrieval is used anywhere in this notebook. `openai`/NVIDIA are deliberately **not** installed - this notebook never imports `openai` and never calls NVIDIA/DeepSeek.

In [ ]:
!pip install -q torch transformers sentence-transformers bitsandbytes accelerate groq pandas numpy matplotlib scikit-learn huggingface_hub tabulate

## Section 4: Kaggle Secrets / Groq API Setup

This experiment's judge is the project's **real production Groq judge** (`model_name="groq_judge"` -> `openai/gpt-oss-120b` via `llm/groq_client.py`, unmodified) - the same judge backend the official Qwen+RAG benchmark used. NVIDIA is not used anywhere in this notebook.

```
Kaggle Secrets -> notebook -> os.environ["GROQ_API_KEY"] -> KidsNutriLLMClient (Section 7) -> KidsNutriGroqClient (Section 11) -> groq_judge
```

**Required Kaggle secret name:** `GROQ_API_KEY`. **Optional:** `HF_TOKEN` (only helps avoid Hugging Face Hub rate limits when loading Qwen; the model itself is not gated).

This cell only ever prints whether the key loaded (`YES`/`NO`) - never the value - and does not write a `.env` file or any other persistent plaintext copy of a secret. The key is never hardcoded in this notebook.

In [ ]:
def _load_kaggle_secret(client, name):
    try:
        return client.get_secret(name)
    except Exception:
        return None


_hf_token = None

if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()

    _groq_key = _load_kaggle_secret(_secrets, "GROQ_API_KEY")
    _hf_token = _load_kaggle_secret(_secrets, "HF_TOKEN")  # optional

    if _groq_key:
        os.environ["GROQ_API_KEY"] = _groq_key
else:
    print("Not running on Kaggle - expecting GROQ_API_KEY to already be set in the local environment.")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged into HuggingFace Hub via Kaggle Secrets.")

# Never print actual secret values - only whether they were loaded.
print(f"GROQ key loaded: {'YES' if os.environ.get('GROQ_API_KEY') else 'NO'}")

if not os.environ.get("GROQ_API_KEY"):
    print("ERROR: GROQ_API_KEY not set (Kaggle Secrets, Add-ons -> Secrets). This experiment's judge "
          "(groq_judge -> openai/gpt-oss-120b) requires it. NVIDIA is not used anywhere in this notebook.")

## Section 5: Load Benchmark Dataset

Make the extracted project the active working directory, put it on `sys.path`, and load the finalized 49-case dataset (`evaluation/dataset.py`, unmodified) - the same dataset the official Qwen+RAG benchmark uses. This notebook only ever reads a curated subset of these 49 cases (Section 6); it never adds, edits, or invents a case.

In [ ]:
import sys

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Current working directory: {os.getcwd()}")

from evaluation.dataset import EVALUATION_DATA

print(f"Finalized evaluation dataset loaded: {len(EVALUATION_DATA)} cases "
      f"({EVALUATION_DATA[0]['id']} .. {EVALUATION_DATA[-1]['id']})")
assert len(EVALUATION_DATA) == 49, f"Expected 49 finalized cases, found {len(EVALUATION_DATA)}."
print("Confirmed: this is the same finalized 49-case dataset used by the official Qwen+RAG benchmark "
      "(docs/evaluation/phase2c_gold_annotations.json) - not a new or modified dataset.")

# Defined here (early) rather than only at save-time, so the checkpointed
# evaluation loop in Section 14 has somewhere to write
# experiment1_no_rag_checkpoint.json as it goes, before the final Section 15
# save-files cell runs. Reused unchanged later for the final CSV/JSON/PNG outputs.
OUTPUT_DIR = os.path.join(project_root, "reports")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")

## Section 6: Verify Exact 30-Case Selection

The **same locked, curated 30-case subset** used for the official Qwen+RAG benchmark - not a new random sample. Every assertion below must pass before any generation or judge call happens.

In [ ]:
SELECTED_30_IDS = [
    "EVAL_001", "EVAL_003", "EVAL_004", "EVAL_005", "EVAL_006",
    "EVAL_008", "EVAL_010", "EVAL_011", "EVAL_013", "EVAL_014",
    "EVAL_016", "EVAL_020", "EVAL_021", "EVAL_023", "EVAL_025",
    "EVAL_026", "EVAL_037", "EVAL_028", "EVAL_029", "EVAL_031",
    "EVAL_035", "EVAL_036", "EVAL_038", "EVAL_039", "EVAL_041",
    "EVAL_043", "EVAL_044", "EVAL_046", "EVAL_048", "EVAL_049",
]

assert len(SELECTED_30_IDS) == 30, f"Expected exactly 30 selected IDs, got {len(SELECTED_30_IDS)}"
assert len(set(SELECTED_30_IDS)) == 30, "Selected ID list contains duplicates."

_original_49_ids = {c["id"] for c in EVALUATION_DATA}
assert len(_original_49_ids) == 49, f"Master dataset is not 49 cases (found {len(_original_49_ids)})."
_outside_49 = set(SELECTED_30_IDS) - _original_49_ids
assert not _outside_49, f"Selected IDs not present in the original 49-case dataset: {sorted(_outside_49)}"

# Order cases by SELECTED_30_IDS (not EVALUATION_DATA's own order), so both
# configurations iterate in exactly the same, exactly this, sequence.
_by_id = {c["id"]: c for c in EVALUATION_DATA}
dataset_30 = [_by_id[cid] for cid in SELECTED_30_IDS]

assert len(dataset_30) == 30, f"Expected 30 matching cases, found {len(dataset_30)}"
assert [c["id"] for c in dataset_30] == SELECTED_30_IDS, "dataset_30 ordering does not match SELECTED_30_IDS."

n_rag_applicable_30 = sum(1 for c in dataset_30 if c.get("relevant_chunk_ids") is not None)
n_non_rag_30 = len(dataset_30) - n_rag_applicable_30
assert n_rag_applicable_30 == 25, f"Expected 25 RAG-applicable cases among the selected 30, found {n_rag_applicable_30}."
assert n_non_rag_30 == 5, f"Expected 5 non-RAG cases among the selected 30, found {n_non_rag_30}."

print("EXPERIMENT 1 BENCHMARK: 30 curated cases (same locked selection as the official Qwen+RAG benchmark)")
print(f"Selected IDs ({len(SELECTED_30_IDS)}): {SELECTED_30_IDS}")
print(f"RAG-applicable within the 30: {n_rag_applicable_30}")
print(f"Non-RAG within the 30: {n_non_rag_30}")
print("All pre-run assertions passed: 30 unique cases, exact ID match, all IDs verified inside the "
      "original 49, 25 RAG-applicable / 5 non-RAG confirmed.")

## Section 7: Load Qwen2.5-7B-Instruct and Diet Planner

Loads the SAME production answer-generation model, via the SAME `KidsNutriLLMClient` (`llm/llm_client.py`, unmodified) and the same `_call_local_transformers` path (4-bit quantized) the official benchmark uses. This same `client` object is reused later (Section 11) as the Groq judge's `llm_client` too - exactly how `evaluation/evaluator.py` shares one `llm_client` between generation and judging.

Also constructs the real, unmodified `DietPlanner`/`KidsNutriDatabase` (`planner/diet_planner.py`) - Configuration A still needs it, since the Diet Planner is not RAG; it is a separate deterministic subsystem.

**No RAG retriever is constructed here** - this notebook never runs RAG retrieval (Configuration B's answers already exist from the official benchmark), so `rag/retriever.py`/FAISS are not loaded at all.

In [ ]:
from llm.llm_client import KidsNutriLLMClient
from planner.diet_planner import KidsNutriDatabase, DietPlanner

client = KidsNutriLLMClient()  # default_model="qwen_local"

print("=== Qwen local answer-generation smoke test ===")
print("Model: Qwen/Qwen2.5-7B-Instruct (local Hugging Face Transformers, 4-bit, via KidsNutriLLMClient._call_local_transformers)")
print("No API call is involved in this step - inference runs entirely on-GPU.")

_smoke_system = "You are a concise pediatric nutrition assistant."
_smoke_user = "In one short sentence, why is iron important for a growing child?"

response, latency = client.generate_response(_smoke_system, _smoke_user, model_name="qwen_local")
print(f"\nResponse: {response}")
print(f"Latency: {latency:.2f}s")
print("\nConfirmed: GPU detected, tokenizer + model loaded, output produced via the project's real generation path.")
print("This same 'client' object is reused in Section 11 as the Groq judge's llm_client too.")

print("\n=== Diet Planner initialization ===")
db = KidsNutriDatabase()
planner = DietPlanner(db)
print("Confirmed: planner (planner/diet_planner.py) initialized - real, unmodified production code.")

## Section 8: Run Qwen WITHOUT RAG (Configuration A)

`Question -> Qwen2.5-7B-Instruct -> Answer`. No `retriever.retrieve()` call, no RAG documents in the prompt. The Diet Planner still runs (it is a separate, non-RAG, deterministic subsystem) so the same profile information is available as in the official benchmark. Uses `generate_llm_prompt()` (`llm/prompt_templates.py`, unmodified) - the SAME function the official benchmark uses - with an empty list passed as the RAG context.

`config_a_results` is produced exactly once here and is **never regenerated** later in this notebook.

In [ ]:
from llm.prompt_templates import generate_llm_prompt

print("=== CONFIGURATION A: Qwen2.5-7B-Instruct WITHOUT RAG ===")
print(f"Running {len(dataset_30)} cases. No FAISS/BM25 retrieval, no RAG context in the prompt.\n")

config_a_results = []
for i, case in enumerate(dataset_30, 1):
    q_id = case["id"]
    question = case["question"]
    profile = case.get("profile") or {}
    print(f"  [{i}/{len(dataset_30)}] {q_id}...")

    # Same deterministic planner, same profile handling as the official
    # Qwen+RAG benchmark (evaluation/evaluator.py).
    plan = planner.generate_meal_plan(profile)

    # generate_llm_prompt() is the SAME production function used by the
    # official benchmark. With an empty list, the "Retrieved Pediatric
    # Nutrition Guidelines (RAG Context)" section of the template is present
    # but has zero entries - no RAG documents are ever shown to the model.
    system_prompt, user_prompt = generate_llm_prompt(plan, [], query=question)

    response, latency = client.generate_response(system_prompt, user_prompt, model_name="qwen_local")

    config_a_results.append({
        "case_id": q_id,
        "question": question,
        "generated_answer": response,
        "model_name": "qwen2.5-7b-instruct",
        "rag_enabled": False,
        "retrieved_chunk_ids": [],
        "retrieved_context": "",
        "planner_output": plan,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "temperature": client.gen_config["temperature"],
        "top_p": client.gen_config["top_p"],
        "max_new_tokens": client.gen_config["max_new_tokens"],
        "latency_seconds": latency,
    })

print(f"\nConfiguration A complete: {len(config_a_results)}/{len(dataset_30)} answers generated.")

## Section 9: Validate Configuration A Outputs

Structural checks before spending any Groq quota: correct case count, correct/matching case ordering, and no RAG leakage into Configuration A.

In [ ]:
print("=== Section 9: Output Validation ===")

assert len(config_a_results) == 30, f"Configuration A produced {len(config_a_results)} results, expected 30."
print("Confirmed: Configuration A produced exactly 30 results.")

a_ids = [r["case_id"] for r in config_a_results]
assert a_ids == SELECTED_30_IDS, "Configuration A's case order does not match SELECTED_30_IDS."
print("Confirmed: Configuration A iterated the exact same 30 case IDs, in the exact same order, as SELECTED_30_IDS.")

non_empty = [r["case_id"] for r in config_a_results if r["retrieved_chunk_ids"]]
assert not non_empty, f"Configuration A (no-RAG) unexpectedly has retrieved chunks for: {non_empty}"
print("Confirmed: Configuration A has zero retrieved chunks for every case (no RAG leakage).")

print("\nAll output validation checks passed.")

## Section 10: Existing Official Qwen + RAG Benchmark Results

**These values are NOT recomputed by this notebook.** They are the already-measured results from the prior official 30-case Qwen+RAG Kaggle benchmark run, entered here exactly once as the single source of truth for the RAG side of the final comparison. This notebook only ever produces the missing No-RAG numbers (Sections 11-15); the two are combined in Section 16.

If the official Qwen+RAG benchmark is ever re-run and produces new numbers, **update this cell** - do not estimate, interpolate, or silently carry forward a stale value.

In [ ]:
# Source: official 30-case Qwen2.5-7B-Instruct + RAG Kaggle benchmark
# (KidsNutriBite_Evaluation.ipynb production run). Entered here exactly once,
# never recomputed or estimated by this notebook.
RAG_OFFICIAL_RESULTS = {
    "Configuration": "Qwen2.5-7B-Instruct + RAG",
    "RAG Enabled": True,
    "Faithfulness": 0.9549,
    "Answer Relevancy": 0.7522,
    "Hallucination Rate": 0.0909,  # 9.09%
}

print("=== Existing Official Qwen + RAG Benchmark Results (not recomputed) ===")
for k, v in RAG_OFFICIAL_RESULTS.items():
    print(f"  {k}: {v}")

## Section 11: Groq Judge Setup

Reuses the **real, unmodified `KidsNutriLLMClient`** (`client`, already constructed in Section 7 for Qwen generation) directly as the judge's `llm_client` - the same class, the same instance, `model_name="groq_judge"` - exactly how `evaluation/evaluator.py` does it for the official benchmark. **No notebook-local adapter class is created**, unlike the earlier NVIDIA version of this notebook.

Also constructs the same embedding model the official benchmark uses for Answer Relevancy (`BAAI/bge-small-en-v1.5`, via `sentence-transformers` directly - functionally identical to `rag/retriever.py`'s `KidsNutriRetriever.model` property, which lazy-loads the exact same `SentenceTransformer(model_name)` call; using it directly here avoids needing to load the RAG/FAISS retrieval stack at all, since this notebook never retrieves).

Retry/backoff, JSON extraction, and empty/malformed-response handling are all reused unchanged from `base_judge.py`'s `call_llm_with_retry`/`safe_parse_json` and `KidsNutriGroqClient`'s own reasoning-budget/pacing configuration (`llm/groq_client.py`, unmodified) - not reimplemented here, and a judge failure never becomes a fabricated score.

In [ ]:
from evaluation.judges.base_judge import BaseJudge
from evaluation.judges.relevancy_judge import RelevancyJudge

if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError("GROQ_API_KEY not set (see Section 4). Cannot proceed without the judge backend.")

print("=== Groq Judge Setup ===")
print("Judge model_name: groq_judge (-> openai/gpt-oss-120b via llm/groq_client.py, unmodified)")
print("llm_client: the SAME KidsNutriLLMClient instance already used for Qwen generation in Section 7.")

baseline_judge = BaseJudge(client, "groq_judge")          # Configuration A grounding - notebook-local prompt, shared retry/parsing infra
relevancy_judge = RelevancyJudge(client, "groq_judge")    # Configuration A relevancy - real, unmodified production class

print("\n=== Embedding model for Answer Relevancy (same model the official benchmark uses) ===")
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("Confirmed: BAAI/bge-small-en-v1.5 loaded directly (same model rag/retriever.py's KidsNutriRetriever.model "
      "property would lazy-load) - no FAISS/RAG index needed, since this notebook never retrieves.")

## Section 12: Baseline Grounding Prompt + Evaluation Function

**Faithfulness / Hallucination (fair-metric design, unchanged from the original Experiment 1 design):** claims in the no-RAG configuration's generated answer are checked against this benchmark's own verified `gold_facts`/`reference_answer` + the Diet Planner output, via a notebook-local prompt (`build_baseline_grounding_prompt`) that mirrors the real `GroundingJudge`'s exact two-source structure and claim schema, with the RAG source honestly relabeled as a verified reference. This is a **baseline factual-grounding evaluation** (`evidence_source = "gold_reference_baseline"`) - a genuinely different evidence source from a retrieved-context faithfulness evaluation, so it is never silently treated as the same measure as the RAG configuration's (already-measured, Section 10) Faithfulness number. The claims schema (`is_supported`, `hallucination_type`, ...) is identical, so the SAME, unmodified `evaluation/metrics/grounding_metrics.py` functions compute Faithfulness/Unsupported-Claim-Rate/Hallucination-type from it.

**Answer Relevancy:** the real, unmodified `RelevancyJudge.generate_hypothetical_questions()`, scored via the same, unmodified `evaluation/metrics/relevancy_metrics.py::calculate_answer_relevancy`, using the SAME embedding model (Section 11) the official benchmark uses.

**Failure handling:** a judge parse/API failure is recorded via `*_status`/`*_judge_failed` and excluded from every average - never converted into a fabricated 0 or 1.

In [ ]:
import json as _json
import evaluation.metrics.grounding_metrics as gm
import evaluation.metrics.relevancy_metrics as relm


def build_baseline_grounding_prompt(question, generated_answer, gold_facts, reference_answer, planner_output):
    """
    Notebook-local prompt only - evaluation/judges/grounding_judge.py is not
    modified (and is not even imported in this notebook, since Configuration
    B is not evaluated here). Mirrors GroundingJudge.evaluate_grounding's
    exact two-source structure and output schema, with SOURCE 1 honestly
    relabeled as a verified reference (gold_facts + reference_answer)
    instead of RAG retrieved context, since the answer being checked was
    generated WITHOUT retrieval access.
    """
    if gold_facts:
        facts_str = "\n".join(f"- {fact}" for fact in gold_facts)
    else:
        facts_str = "None"
    if reference_answer:
        facts_str += f"\n\nReference answer: {reference_answer}"

    planner_str = _json.dumps(planner_output, indent=2) if planner_output else "None"

    return f"""
You are an expert pediatric clinical reviewer.
Your task is to deconstruct a pediatric chatbot's answer into individual factual claims and verify them against the provided source materials to evaluate baseline factual grounding.

METHODOLOGICAL NOTE: the GENERATED ANSWER below was produced WITHOUT retrieval access - the model that wrote it had no RAG context available, only SOURCE 1 and SOURCE 2 below. SOURCE 1 here is a VERIFIED REFERENCE (this benchmark's independently gold-annotated facts and reference answer), NOT retrieved context. This is a baseline factual-grounding evaluation, not a retrieved-context faithfulness evaluation - do not treat it as measuring the same thing as a RAG faithfulness score.

QUESTION: "{question}"

SOURCE 1 (Verified Reference Facts - NOT retrieved context):
{facts_str}

SOURCE 2 (Diet Planner Structured Output):
{planner_str}

GENERATED ANSWER (produced without retrieval access):
"{generated_answer}"

TASK:
1. Deconstruct the GENERATED ANSWER into a list of atomic, individual factual claims.
2. Assign each claim a unique ID (e.g., "C001", "C002").
3. For each claim, check if it is explicitly supported by SOURCE 1 (Verified Reference) or SOURCE 2 (Planner). Set `is_supported` to true or false.
4. Set `support_source` to "Reference", "Planner", "Both", or "None".
5. If `is_supported` is false, you MUST include a key `hallucination_type`. The value must be either:
   - "Intrinsic": The claim directly contradicts information provided in the sources.
   - "Extrinsic": The claim is an unverified addition not present in the sources.

Output a JSON object with a single key "claims".
Example:
{{
  "claims": [
    {{
      "claim_id": "C001",
      "claim": "Iron-rich foods support red blood cell production.",
      "is_supported": true,
      "support_source": "Reference"
    }},
    {{
      "claim_id": "C002",
      "claim": "Eggs contain protein.",
      "is_supported": true,
      "support_source": "Planner"
    }},
    {{
      "claim_id": "C003",
      "claim": "Give the child 5mg of iron supplement daily.",
      "is_supported": false,
      "support_source": "None",
      "hallucination_type": "Extrinsic"
    }}
  ]
}}
"""


def evaluate_baseline_grounding(base_judge, question, generated_answer, gold_facts, reference_answer, planner_output, q_id, max_retries=3):
    prompt = build_baseline_grounding_prompt(question, generated_answer, gold_facts, reference_answer, planner_output)
    return base_judge.call_llm_with_retry(prompt, max_retries=max_retries, q_id=q_id, question=question, model_response=generated_answer)


def evaluate_single_case_no_rag(r, case):
    """Evaluates ONE Configuration A (no-RAG) case: grounding + relevancy.
    Returns the same per-case dict shape used throughout this project's
    evaluation notebooks. Never converts a judge failure into a fabricated
    score - see the *_status / *_judge_failed fields."""
    q_id = r["case_id"]
    question = r["question"]
    answer = r["generated_answer"]

    gold_facts = [f.get("fact_text") for f in (case.get("gold_facts") or []) if f.get("fact_text")]
    reference_answer = case.get("reference_answer", "")
    grounding_data = evaluate_baseline_grounding(
        baseline_judge, question, answer, gold_facts, reference_answer, r["planner_output"], q_id
    )
    grounding_failed = bool(grounding_data.get("parse_failed"))
    claims = grounding_data.get("claims", [])

    faithfulness_details = gm.calculate_faithfulness_details(claims, evaluation_failed=grounding_failed)
    unsupported_details = gm.calculate_unsupported_claim_rate_details(claims, evaluation_failed=grounding_failed)
    hallu_type_details = gm.calculate_response_hallucination_type_details(claims, evaluation_failed=grounding_failed)
    if unsupported_details["status"] in (gm.UNSUPPORTED_CLAIM_RATE_STATUS_VALID, gm.UNSUPPORTED_CLAIM_RATE_STATUS_REAL_ZERO):
        is_hallucinated = unsupported_details["score"] > 0
    else:
        is_hallucinated = None

    relevancy_data = relevancy_judge.generate_hypothetical_questions(answer, num_questions=3, q_id=q_id)
    relevancy_failed = bool(relevancy_data.get("parse_failed"))
    questions_list = relevancy_data.get("generated_questions", [])
    relevancy_scores = relm.calculate_answer_relevancy(question, questions_list, embedding_model, evaluation_failed=relevancy_failed)

    return {
        "case_id": q_id,
        "rag_enabled": False,
        "evidence_source": "gold_reference_baseline",
        "faithfulness": faithfulness_details["score"],
        "faithfulness_status": faithfulness_details["status"],
        "unsupported_claim_rate": unsupported_details["score"],
        "unsupported_claim_rate_status": unsupported_details["status"],
        "is_hallucinated": is_hallucinated,
        "hallucination_type_status": hallu_type_details["status"],
        "has_intrinsic_claim": hallu_type_details["has_intrinsic"],
        "has_extrinsic_claim": hallu_type_details["has_extrinsic"],
        "answer_relevancy": relevancy_scores["mean_similarity"],
        "answer_relevancy_status": relevancy_scores["status"],
        "claims_total_count": faithfulness_details["total_count"],
        "grounding_judge_failed": grounding_failed,
        "relevancy_judge_failed": relevancy_failed,
    }


dataset_30_by_id = {c["id"]: c for c in dataset_30}
config_a_results_by_id = {r["case_id"]: r for r in config_a_results}
print(f"Evaluation function ready. {len(dataset_30_by_id)} cases indexed by ID.")
print("Proceed to Section 13 (smoke tests) before running the full 30-case evaluation.")

## Section 13: Smoke Tests (EVAL_001)

Before spending 30 x 2 = up to 60 Groq API calls, verify both judge paths work on a SINGLE already-generated case (`EVAL_001`) - **no Qwen regeneration**, `config_a_results` is only read here. Each smoke test uses `max_retries=1` so a real failure surfaces immediately instead of waiting through the full retry/backoff schedule. **The full 30-case evaluation (Section 14) does not run unless both smoke tests below pass** - each one raises and stops the notebook on failure.

In [ ]:
smoke_case_id = "EVAL_001"
smoke_r = config_a_results_by_id[smoke_case_id]
smoke_case = dataset_30_by_id[smoke_case_id]

smoke_tests_passed = {"baseline_grounding": False, "relevancy": False}

# --- A. Baseline grounding smoke test ---
print("=" * 70)
print("SMOKE TEST A: Baseline Grounding Judge (EVAL_001, no-RAG evidence)")
print("=" * 70)
gold_facts_smoke = [f.get("fact_text") for f in (smoke_case.get("gold_facts") or []) if f.get("fact_text")]
reference_answer_smoke = smoke_case.get("reference_answer", "")
result_a = evaluate_baseline_grounding(
    baseline_judge, smoke_r["question"], smoke_r["generated_answer"],
    gold_facts_smoke, reference_answer_smoke, smoke_r["planner_output"],
    smoke_case_id, max_retries=1
)
parse_failed_a = bool(result_a.get("parse_failed"))
claims_a = result_a.get("claims", [])
print(f"\nSMOKE TEST RESULT (A)")
print(f"parse_failed = {parse_failed_a}")
print(f"claims count = {len(claims_a)}")
print(f"first claim = {claims_a[0] if claims_a else None}")
print(f"response status = {'FAILED' if parse_failed_a else 'OK'}")
smoke_tests_passed["baseline_grounding"] = (not parse_failed_a) and len(claims_a) > 0

if not smoke_tests_passed["baseline_grounding"]:
    raise RuntimeError("Smoke test A (baseline grounding) FAILED - stopping before the 30-case run. "
                        "Inspect reports/judge_parse_failures.csv and reports/judge_raw_outputs.log.")
print("\nSmoke test A PASSED.\n")

# --- B. Answer Relevancy smoke test ---
print("=" * 70)
print("SMOKE TEST B: Relevancy Judge (EVAL_001)")
print("=" * 70)
result_b = relevancy_judge.generate_hypothetical_questions(smoke_r["generated_answer"], num_questions=3, q_id=smoke_case_id)
parse_failed_b = bool(result_b.get("parse_failed"))
questions_b = result_b.get("generated_questions", [])
print(f"\nSMOKE TEST RESULT (B)")
print(f"parse_failed = {parse_failed_b}")
print(f"claims count = {len(questions_b)} (generated_questions)")
print(f"first claim = {questions_b[0] if questions_b else None}")
print(f"response status = {'FAILED' if parse_failed_b else 'OK'}")
smoke_tests_passed["relevancy"] = (not parse_failed_b) and len(questions_b) > 0

if not smoke_tests_passed["relevancy"]:
    raise RuntimeError("Smoke test B (relevancy) FAILED - stopping before the 30-case run. "
                        "Note: RelevancyJudge.generate_hypothetical_questions() hardcodes max_retries=3 "
                        "internally (evaluation/judges/relevancy_judge.py, not modified here) - a failure "
                        "here has already exhausted 3 attempts.")
print("\nSmoke test B PASSED.\n")

print("=" * 70)
print(f"ALL SMOKE TESTS PASSED: {smoke_tests_passed}")
print("Proceed to Section 14 to run the full 30-case evaluation.")
print("=" * 70)

## Section 14: Full No-RAG Evaluation (Checkpointed)

Only reached if both smoke tests in Section 13 passed. Evaluates **only** `config_a_results` (Configuration A / no-RAG) - `evaluate_config(..., rag_enabled=True)` is never called and `config_b_results` does not exist in this notebook at all.

**Checkpointing:** results are saved to `reports/experiment1_no_rag_checkpoint.json` after **every single case**. Re-running this cell (after an interrupted run, a Kaggle session restart, a rate-limit wait, etc.) loads whatever is already in the checkpoint and skips those case IDs, so Groq quota is never spent twice on the same case. Delete the checkpoint file manually if you deliberately want to force a full re-evaluation from scratch.

In [ ]:
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "experiment1_no_rag_checkpoint.json")


def _load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return _json.load(f)
    return []


def _save_checkpoint(results):
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        _json.dump(results, f, indent=2)


assert all(smoke_tests_passed.values()), (
    f"Not all smoke tests passed ({smoke_tests_passed}) - do not run the full evaluation. Re-check Section 13."
)

config_a_eval = _load_checkpoint()
done_ids = {r["case_id"] for r in config_a_eval}
print(f"Checkpoint loaded from {CHECKPOINT_PATH}: {len(config_a_eval)}/{len(config_a_results)} cases already evaluated.")

print("\n=== Evaluating Configuration A (No-RAG baseline) - Groq judge ===")
for i, r in enumerate(config_a_results, 1):
    q_id = r["case_id"]
    if q_id in done_ids:
        print(f"  [{i}/{len(config_a_results)}] {q_id} - already in checkpoint, skipping.")
        continue

    print(f"  [{i}/{len(config_a_results)}] {q_id}...")
    case = dataset_30_by_id[q_id]
    case_result = evaluate_single_case_no_rag(r, case)
    config_a_eval.append(case_result)
    done_ids.add(q_id)
    _save_checkpoint(config_a_eval)
    print(f"      done, checkpoint saved ({len(config_a_eval)}/{len(config_a_results)}).")

assert len(config_a_eval) == 30, f"Expected 30 evaluated cases, got {len(config_a_eval)}."
config_a_eval_ids = [e["case_id"] for e in config_a_eval]
assert set(config_a_eval_ids) == set(SELECTED_30_IDS), "config_a_eval case IDs do not match SELECTED_30_IDS."

print(f"\nEvaluation complete: {len(config_a_eval)} Config-A cases judged (Groq, groq_judge -> openai/gpt-oss-120b).")

## Section 15: Save No-RAG Output Files

Writes the three required no-RAG output files to `reports/`. No value is fabricated: a case with a failed evaluation keeps its real status/failure flag in every saved file. Aggregation follows the same philosophy as the official benchmark's own comparator (`evaluation/comparator.py`): averages are computed **only** over valid (non-failed) cases, with failure counts reported alongside as their own fields.

In [ ]:
import pandas as pd

# --- experiment1_no_rag_answers.csv ---
answers_records = []
for r in config_a_results:
    answers_records.append({
        "case_id": r["case_id"],
        "question": r["question"],
        "generated_answer": r["generated_answer"],
        "model_name": r["model_name"],
        "rag_enabled": r["rag_enabled"],
        "temperature": r["temperature"],
        "top_p": r["top_p"],
        "max_new_tokens": r["max_new_tokens"],
        "latency_seconds": r["latency_seconds"],
    })
answers_df = pd.DataFrame(answers_records)
answers_path = os.path.join(OUTPUT_DIR, "experiment1_no_rag_answers.csv")
answers_df.to_csv(answers_path, index=False)
print(f"Saved: {answers_path} ({len(answers_df)} rows)")

# --- experiment1_no_rag_metrics.csv ---
metrics_df = pd.DataFrame(config_a_eval)
metrics_path = os.path.join(OUTPUT_DIR, "experiment1_no_rag_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)
print(f"Saved: {metrics_path} ({len(metrics_df)} rows)")

# --- experiment1_no_rag_summary.json ---
_valid_faith = metrics_df[metrics_df["faithfulness"].notna()]["faithfulness"]
_valid_relevancy = metrics_df[metrics_df["answer_relevancy"].notna()]["answer_relevancy"]
_valid_hallu = metrics_df[metrics_df["is_hallucinated"].notna()]["is_hallucinated"]

no_rag_summary = {
    "Configuration": "Qwen2.5-7B-Instruct",
    "RAG Enabled": False,
    "Faithfulness": round(float(_valid_faith.mean()), 4) if len(_valid_faith) > 0 else None,
    "Faithfulness Valid Cases": int(len(_valid_faith)),
    "Answer Relevancy": round(float(_valid_relevancy.mean()), 4) if len(_valid_relevancy) > 0 else None,
    "Answer Relevancy Valid Cases": int(len(_valid_relevancy)),
    "Hallucination Rate": round(float(_valid_hallu.mean()), 4) if len(_valid_hallu) > 0 else None,
    "Hallucination Rate Valid Cases": int(len(_valid_hallu)),
    "Total Cases": int(len(metrics_df)),
    "Grounding Judge Failures": int(metrics_df["grounding_judge_failed"].sum()),
    "Relevancy Judge Failures": int(metrics_df["relevancy_judge_failed"].sum()),
}

summary_path = os.path.join(OUTPUT_DIR, "experiment1_no_rag_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    _json.dump(no_rag_summary, f, indent=2)
print(f"Saved: {summary_path}")
print("\n=== No-RAG Summary ===")
for k, v in no_rag_summary.items():
    print(f"  {k}: {v}")

## Section 16: Final RAG vs No-RAG Comparison

Combines the just-measured no-RAG numbers (Section 15) with the existing official RAG numbers (Section 10, not recomputed) into `experiment1_rag_vs_no_rag_metrics.csv`. Nothing here is invented - if a no-RAG metric is missing (all cases failed for that metric), it is written as `None`/blank, never a fabricated 0.

In [ ]:
comparison_rows = [
    {
        "Configuration": "Qwen2.5-7B-Instruct",
        "Faithfulness": no_rag_summary["Faithfulness"],
        "Answer Relevancy": no_rag_summary["Answer Relevancy"],
        "Hallucination Rate": no_rag_summary["Hallucination Rate"],
    },
    {
        "Configuration": RAG_OFFICIAL_RESULTS["Configuration"],
        "Faithfulness": RAG_OFFICIAL_RESULTS["Faithfulness"],
        "Answer Relevancy": RAG_OFFICIAL_RESULTS["Answer Relevancy"],
        "Hallucination Rate": RAG_OFFICIAL_RESULTS["Hallucination Rate"],
    },
]
comparison_df = pd.DataFrame(comparison_rows)

comparison_path = os.path.join(OUTPUT_DIR, "experiment1_rag_vs_no_rag_metrics.csv")
comparison_df.to_csv(comparison_path, index=False)
print(f"Saved: {comparison_path}")

print("\n=== Experiment 1: RAG vs No-RAG Comparison ===")
print(comparison_df.to_string(index=False))

## Section 17: Publication-Friendly Graphs

Faithfulness and Answer Relevancy (higher is better) are grouped in one chart. Hallucination Rate (lower is better) is deliberately kept in a **separate** chart, since mixing a "higher is better" and a "lower is better" metric in one visual is misleading. No green/red "good/bad" color coding is used.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})

config_labels = comparison_df["Configuration"].tolist()
bar_colors = ["#5b8db8", "#e8a13c"]  # neutral blue/amber - not a good/bad color pair
bar_hatches = ["", "//"]

# --- Chart 1: Faithfulness + Answer Relevancy (higher is better) ---
metrics_higher_better = ["Faithfulness", "Answer Relevancy"]
x = np.arange(len(metrics_higher_better))
width = 0.35

fig, ax = plt.subplots(figsize=(6, 4.5))
for i, label in enumerate(config_labels):
    values = [comparison_df.loc[comparison_df["Configuration"] == label, m].values[0] for m in metrics_higher_better]
    values = [v if v is not None else 0.0 for v in values]
    bars = ax.bar(x + (i - 0.5) * width, values, width, label=label, hatch=bar_hatches[i],
                   edgecolor="black", linewidth=0.8, color=bar_colors[i])
    ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metrics_higher_better)
ax.set_ylabel("Score (higher is better)")
ax.set_ylim(0, 1.05)
ax.set_title("Faithfulness and Answer Relevancy: Base Model vs. RAG")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()

chart1_path = os.path.join(OUTPUT_DIR, "experiment1_metric_comparison.png")
fig.savefig(chart1_path, dpi=300, bbox_inches="tight")
print(f"Saved: {chart1_path}")
plt.show()

# --- Chart 2: Hallucination Rate (lower is better) - kept visually SEPARATE ---
fig2, ax2 = plt.subplots(figsize=(4.5, 4.5))
hallu_values = [comparison_df.loc[comparison_df["Configuration"] == label, "Hallucination Rate"].values[0] for label in config_labels]
hallu_values_plot = [v if v is not None else 0.0 for v in hallu_values]
bars2 = ax2.bar(config_labels, hallu_values_plot, width=0.5, edgecolor="black", linewidth=0.8,
                 color=bar_colors, hatch=bar_hatches)
ax2.bar_label(bars2, fmt="%.3f", padding=2, fontsize=9)
ax2.set_ylabel("Hallucination Rate (lower is better)")
ax2.set_ylim(0, 1.05)
ax2.set_title("Hallucination Rate: Base Model vs. RAG")
ax2.set_xticks(range(len(config_labels)))
ax2.set_xticklabels(config_labels, rotation=10, ha="right")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
fig2.tight_layout()

chart2_path = os.path.join(OUTPUT_DIR, "experiment1_hallucination_rate.png")
fig2.savefig(chart2_path, dpi=300, bbox_inches="tight")
print(f"Saved: {chart2_path}")
plt.show()

print("\n=== Exact numeric values used in the graphs above ===")
print(comparison_df.to_string(index=False))

## Section 18: Final Summary

In [ ]:
print("\n" + "=" * 78)
print("FINAL COMPARISON TABLE")
print("=" * 78)
print(f"{'Configuration':<32} {'Faithfulness':>14} {'Answer Relevancy':>18} {'Hallucination Rate':>20}")
for row in comparison_rows:
    faith = f"{row['Faithfulness']:.4f}" if row["Faithfulness"] is not None else "N/A"
    rel = f"{row['Answer Relevancy']:.4f}" if row["Answer Relevancy"] is not None else "N/A"
    hallu = f"{row['Hallucination Rate']:.4f}" if row["Hallucination Rate"] is not None else "N/A"
    print(f"{row['Configuration']:<32} {faith:>14} {rel:>18} {hallu:>20}")

print("\nEXPERIMENT 1 (NO-RAG BASELINE) COMPLETE")

total_generations = len(config_a_results)
total_judged = len(config_a_eval)
total_grounding_failures = sum(r["grounding_judge_failed"] for r in config_a_eval)
total_relevancy_failures = sum(r["relevancy_judge_failed"] for r in config_a_eval)

print(f"\nNumber of no-RAG cases processed: {len(dataset_30)}")
print(f"Number of successful generations: {total_generations} / {len(dataset_30)}")
print(f"Number of successful grounding-judge evaluations: {total_judged - total_grounding_failures} / {total_judged}")
print(f"Number of successful relevancy-judge evaluations: {total_judged - total_relevancy_failures} / {total_judged}")
print(f"Number of failed grounding evaluations: {total_grounding_failures}")
print(f"Number of failed relevancy evaluations: {total_relevancy_failures}")
print("\nAll failures are reported explicitly above and in experiment1_no_rag_metrics.csv - "
      "none were converted into a fabricated score.")
print("\nRAG-side numbers (Faithfulness=0.9549, Answer Relevancy=0.7522, Hallucination Rate=9.09%) "
      "are the existing official benchmark results (Section 10) - not recomputed by this notebook.")

if os.environ.get("GROQ_API_KEY") and getattr(client, "groq_client_instance", None) is not None:
    print(f"\nTotal Groq judge tokens used this run (per Groq's own reported usage): "
          f"{client.groq_client_instance.total_tokens_used}")